In [2]:
from pathlib import Path

# Find the FlowCast project root automatically
current = Path.cwd()

BASE_DIR = next(
    (p for p in [current, *current.parents]
     if (p / "models").is_dir()),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError(
        "Could not find the FlowCast folder containing models/"
    )

models_dir = BASE_DIR / "models"

print("Project location:", BASE_DIR)
print("\nModel files:")

for file in sorted(models_dir.iterdir()):
    if file.is_file():
        print(f"{file.name} ({file.stat().st_size:,} bytes)")

Project location: C:\Users\pathi\Desktop\FlowCast

Model files:
accident_risk_model.joblib (64,018,986 bytes)
congestion_classifier.pkl (134,168,345 bytes)
congestion_features.json (191 bytes)
traffic_volume_features.json (338 bytes)
traffic_volume_lstm.keras (87,993 bytes)
traffic_volume_lstm_scaler.pkl (667 bytes)
traffic_volume_random_forest.pkl (558,672,657 bytes)


In [3]:
import json
from pathlib import Path

models_dir = BASE_DIR / "models"

for filename in [
    "traffic_volume_features.json",
    "congestion_features.json",
]:
    path = models_dir / filename
    print(f"\n--- {filename} ---")

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("File not found")


--- traffic_volume_features.json ---
[
  "traffic_volume_lag_1",
  "traffic_volume_lag_2",
  "traffic_volume_lag_4",
  "avg_speed_lag_1",
  "avg_speed_lag_2",
  "traffic_volume_rolling_mean_4",
  "avg_speed_rolling_mean_4",
  "hour",
  "minute",
  "day_of_week",
  "month",
  "is_weekend",
  "is_morning_peak",
  "is_evening_peak"
]

--- congestion_features.json ---
[
  "traffic_volume",
  "avg_speed",
  "occupancy",
  "hour",
  "minute",
  "day_of_week",
  "month",
  "is_weekend",
  "is_morning_peak",
  "is_evening_peak"
]


In [5]:
import joblib

model_files = {
    "Traffic Random Forest": models_dir / "traffic_volume_random_forest.pkl",
    "Congestion Classifier": models_dir / "congestion_classifier.pkl",
    "Accident Risk Model": models_dir / "accident_risk_model.joblib",
}

for name, path in model_files.items():
    print(f"\n{name}")
    try:
        model = joblib.load(path)
        print("Loaded successfully")
        print("Model type:", type(model).__name__)
        print("Expected features:", getattr(model, "n_features_in_", "Not available"))
        print("Classes:", getattr(model, "classes_", "Not applicable"))
    except Exception as e:
        print("Loading error:", str(e))


Traffic Random Forest
Loaded successfully
Model type: RandomForestRegressor
Expected features: 14
Classes: Not applicable

Congestion Classifier
Loaded successfully
Model type: RandomForestClassifier
Expected features: 10
Classes: ['Free-flow' 'Heavy' 'Moderate' 'Severe']

Accident Risk Model
Loaded successfully
Model type: dict
Expected features: Not available
Classes: Not applicable


In [8]:
import json
import joblib

# Inspect accident-risk model
accident_path = models_dir / "accident_risk_model.joblib"
accident_data = joblib.load(accident_path)

print("ACCIDENT MODEL")
if isinstance(accident_data, dict):
    for key, value in accident_data.items():
        print(key, ":", type(value).__name__)
else:
    print(type(accident_data).__name__)

# Compare congestion feature counts
congestion_model = joblib.load(
    models_dir / "congestion_classifier.pkl"
)

with open(models_dir / "congestion_features.json", encoding="utf-8") as f:
    congestion_features = json.load(f)

print("\nCONGESTION FEATURES:", congestion_features)
print("Model expects:", congestion_model.n_features_in_)
print("JSON contains:", len(congestion_features))

ACCIDENT MODEL
model : RandomForestClassifier
imputer : SimpleImputer
features : list
threshold : float

CONGESTION FEATURES: ['traffic_volume', 'avg_speed', 'occupancy', 'hour', 'minute', 'day_of_week', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak']
Model expects: 10
JSON contains: 10


In [9]:
import pandas as pd
import joblib

traffic = pd.read_csv(BASE_DIR / "data" / "traffic_sensor_log.csv")
weather = pd.read_csv(BASE_DIR / "data" / "weather_observations.csv")
calendar = pd.read_csv(BASE_DIR / "data" / "calendar_events.csv")

print("TRAFFIC COLUMNS:")
print(traffic.columns.tolist())

print("\nWEATHER COLUMNS:")
print(weather.columns.tolist())

print("\nCALENDAR COLUMNS:")
print(calendar.columns.tolist())

accident_data = joblib.load(
    BASE_DIR / "models" / "accident_risk_model.joblib"
)
print("\nACCIDENT MODEL FEATURES:")
print(accident_data.get("features", []))
print("Threshold:", accident_data.get("threshold"))

TRAFFIC COLUMNS:
['road_id', 'road_name', 'latitude', 'longitude', 'weather_station_id', 'date', 'time', 'traffic_volume', 'vehicle_count', 'vehicle_type_dist', 'avg_speed', 'occupancy', 'congestion_level', 'travel_time', 'accident_count', 'signal_timing', 'road_capacity']

WEATHER COLUMNS:
['station_id', 'date', 'time', 'weather_condition', 'temperature', 'rainfall', 'visibility']

CALENDAR COLUMNS:
['date', 'public_holiday', 'holiday_name', 'event_flag', 'event_name', 'roadwork_flag']

ACCIDENT MODEL FEATURES:
['latitude', 'longitude', 'traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time', 'signal_timing', 'road_capacity', 'capacity_30min', 'volume_capacity_ratio', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'event_flag', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_l

In [1]:
import joblib
from pathlib import Path

# Use the FlowCast project folder
BASE_DIR = Path(r"C:\Users\pathi\Desktop\FlowCast")

model_path = BASE_DIR / "models" / "accident_risk_model.joblib"

# Load accident risk model
accident_model = joblib.load(model_path)

print("Accident Risk Model Loaded Successfully!")
print("Model Type:", type(accident_model))

if hasattr(accident_model, "n_features_in_"):
    print("Number of Features:", accident_model.n_features_in_)

if hasattr(accident_model, "feature_names_in_"):
    print("Feature Names:")
    print(list(accident_model.feature_names_in_))

if hasattr(accident_model, "classes_"):
    print("Model Classes:", accident_model.classes_)

if hasattr(accident_model, "named_steps"):
    print("Pipeline Steps:", accident_model.named_steps)

Accident Risk Model Loaded Successfully!
Model Type: <class 'dict'>


In [2]:
print("Accident model dictionary keys:")

for key, value in accident_model.items():
    print(f"\nKey: {key}")
    print("Type:", type(value))

    if hasattr(value, "n_features_in_"):
        print("Number of features:", value.n_features_in_)

    if hasattr(value, "feature_names_in_"):
        print("Feature names:", list(value.feature_names_in_))

    if hasattr(value, "classes_"):
        print("Classes:", value.classes_)

Accident model dictionary keys:

Key: model
Type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Number of features: 32
Classes: [0 1]

Key: imputer
Type: <class 'sklearn.impute._base.SimpleImputer'>
Number of features: 32
Feature names: ['latitude', 'longitude', 'traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time', 'signal_timing', 'road_capacity', 'capacity_30min', 'volume_capacity_ratio', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'event_flag', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4']

Key: features
Type: <class 'list'>

Key: threshold
Type: <class 'float'>


In [3]:
print("ACCIDENT MODEL FEATURES")
print("=" * 50)

for i, feature in enumerate(accident_model["features"], start=1):
    print(i, feature)

print("\nTotal features:", len(accident_model["features"]))
print("Threshold:", accident_model["threshold"])
print("Classes:", accident_model["model"].classes_)

ACCIDENT MODEL FEATURES
1 latitude
2 longitude
3 traffic_volume
4 vehicle_count
5 avg_speed
6 occupancy
7 travel_time
8 signal_timing
9 road_capacity
10 capacity_30min
11 volume_capacity_ratio
12 temperature
13 rainfall
14 visibility
15 public_holiday
16 event_flag
17 roadwork_flag
18 hour
19 minute
20 day_of_week
21 day_of_month
22 month
23 is_weekend
24 is_morning_peak
25 is_evening_peak
26 traffic_volume_lag_1
27 traffic_volume_lag_2
28 traffic_volume_lag_4
29 avg_speed_lag_1
30 avg_speed_lag_2
31 traffic_volume_rolling_mean_4
32 avg_speed_rolling_mean_4

Total features: 32
Threshold: 0.1
Classes: [0 1]


In [5]:
import pandas as pd

In [6]:
traffic_file = BASE_DIR / "data" / "traffic_sensor_log.csv"

traffic_df = pd.read_csv(traffic_file)

print("Dataset shape:", traffic_df.shape)
print("\nAll dataset columns:")
for col in traffic_df.columns:
    print(col)

print("\nFirst 5 rows:")
display(traffic_df.head())

Dataset shape: (178468, 17)

All dataset columns:
road_id
road_name
latitude
longitude
weather_station_id
date
time
traffic_volume
vehicle_count
vehicle_type_dist
avg_speed
occupancy
congestion_level
travel_time
accident_count
signal_timing
road_capacity

First 5 rows:


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,avg_speed,occupancy,congestion_level,travel_time,accident_count,signal_timing,road_capacity
0,NL-007,Kingsway @ Park,28.647903,77.149142,WS-NORTH,2025-02-26,07:30,706.0,706,"{""2W"": 0.37, ""Car"": 0.43, ""LCV"": 0.11, ""HCV"": ...",31.2,59.6,Moderate,2.12,0,54,1928
1,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-03-23,21:00,308.0,308,"{""2W"": 0.43, ""Car"": 0.39, ""LCV"": 0.08, ""HCV"": ...",53.5,16.8,Free-flow,2.03,0,32,2327
2,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,11:00,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",49.9,27.5,Free-flow,3.46,0,38,1771
3,NL-006,Central Flyover E,28.622342,77.258257,WS-SOUTH,2025-05-04,01:30,187.0,187,"{""2W"": 0.41, ""Car"": 0.39, ""LCV"": 0.11, ""HCV"": ...",55.1,10.7,Free-flow,2.18,0,30,2285
4,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-01-08,12:30,490.0,490,"{""2W"": 0.39, ""Car"": 0.4, ""LCV"": 0.1, ""HCV"": 0.11}",45.9,27.7,Free-flow,2.37,0,45,2327


In [7]:
weather_file = BASE_DIR / "data" / "weather_observations.csv"
calendar_file = BASE_DIR / "data" / "calendar_events.csv"

weather_df = pd.read_csv(weather_file)
calendar_df = pd.read_csv(calendar_file)

print("WEATHER DATA")
print("Shape:", weather_df.shape)
print("Columns:", list(weather_df.columns))
display(weather_df.head(3))

print("\nCALENDAR DATA")
print("Shape:", calendar_df.shape)
print("Columns:", list(calendar_df.columns))
display(calendar_df.head(3))

WEATHER DATA
Shape: (10872, 7)
Columns: ['station_id', 'date', 'time', 'weather_condition', 'temperature', 'rainfall', 'visibility']


,station_id,date,time,weather_condition,temperature,rainfall,visibility
0,WS-NORTH,01/01/2025,00:00,Clear,12.7,0.0,10000.0
1,WS-NORTH,01/01/2025,01:00,Clear,8.8,0.0,10000.0
2,WS-NORTH,01/01/2025,02:00,clear,9.8,0.0,10000.0



CALENDAR DATA
Shape: (151, 6)
Columns: ['date', 'public_holiday', 'holiday_name', 'event_flag', 'event_name', 'roadwork_flag']


,date,public_holiday,holiday_name,event_flag,event_name,roadwork_flag
0,2025-01-01,1,New Year's Day,0,NaN,0
1,2025-01-02,0,NaN,0,NaN,0
2,2025-01-03,0,NaN,0,NaN,0


In [8]:
# Check duplicate weather records
print("Weather duplicate records:")
print(
    weather_df.duplicated(
        subset=["station_id", "date", "time"]
    ).sum()
)

# Check duplicate calendar dates
print("\nCalendar duplicate dates:")
print(calendar_df["date"].duplicated().sum())

# Check missing values
print("\nWeather missing values:")
print(weather_df.isnull().sum())

print("\nCalendar missing values:")
print(calendar_df.isnull().sum())

# Check traffic and weather join columns
print("\nTraffic weather station IDs:")
print(traffic_df["weather_station_id"].unique()[:10])

print("\nWeather station IDs:")
print(weather_df["station_id"].unique()[:10])

Weather duplicate records:
0

Calendar duplicate dates:
0

Weather missing values:
station_id             0
date                   0
time                   0
weather_condition      0
temperature          167
rainfall               0
visibility           111
dtype: int64

Calendar missing values:
date                0
public_holiday      0
holiday_name      145
event_flag          0
event_name        145
roadwork_flag       0
dtype: int64

Traffic weather station IDs:
['WS-NORTH' 'WS-CENTRAL' 'WS-SOUTH']

Weather station IDs:
['WS-NORTH' 'WS-CENTRAL' 'WS-SOUTH']


In [9]:
# Make copies so original datasets remain unchanged
traffic_clean = traffic_df.copy()
weather_clean = weather_df.copy()
calendar_clean = calendar_df.copy()

# Convert date columns
traffic_clean["date"] = pd.to_datetime(
    traffic_clean["date"], errors="coerce"
).dt.date

weather_clean["date"] = pd.to_datetime(
    weather_clean["date"], errors="coerce"
).dt.date

calendar_clean["date"] = pd.to_datetime(
    calendar_clean["date"], errors="coerce"
).dt.date

# Standardize time format
traffic_clean["time"] = pd.to_datetime(
    traffic_clean["time"].astype(str),
    format="%H%M",
    errors="coerce"
).dt.strftime("%H:%M")

weather_clean["time"] = pd.to_datetime(
    weather_clean["time"].astype(str),
    format="%H%M",
    errors="coerce"
).dt.strftime("%H:%M")

# Fill missing weather values
weather_clean["temperature"] = weather_clean["temperature"].fillna(
    weather_clean["temperature"].median()
)

weather_clean["visibility"] = weather_clean["visibility"].fillna(
    weather_clean["visibility"].median()
)

# Fill missing calendar names
calendar_clean["holiday_name"] = calendar_clean["holiday_name"].fillna("No Holiday")
calendar_clean["event_name"] = calendar_clean["event_name"].fillna("No Event")

# Merge traffic with weather
merged_df = traffic_clean.merge(
    weather_clean,
    left_on=["weather_station_id", "date", "time"],
    right_on=["station_id", "date", "time"],
    how="left"
)

# Merge calendar information
merged_df = merged_df.merge(
    calendar_clean,
    on="date",
    how="left"
)

print("Merged dataset shape:", merged_df.shape)
print("\nMissing values after merging:")
print(merged_df.isnull().sum())

display(merged_df.head())

Merged dataset shape: (858164, 27)

Missing values after merging:
road_id                    0
road_name                  0
latitude                   0
longitude                  0
weather_station_id         0
date                       0
time                  858164
traffic_volume         21269
vehicle_count              0
vehicle_type_dist          0
avg_speed              21195
occupancy              21219
congestion_level      127715
travel_time                0
accident_count             0
signal_timing              0
road_capacity              0
station_id            148916
weather_condition     148916
temperature           148916
rainfall              148916
visibility            148916
public_holiday             0
holiday_name               0
event_flag                 0
event_name                 0
roadwork_flag              0
dtype: int64


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,...,station_id,weather_condition,temperature,rainfall,visibility,public_holiday,holiday_name,event_flag,event_name,roadwork_flag
0,NL-007,Kingsway @ Park,28.647903,77.149142,WS-NORTH,2025-02-26,NaN,706.0,706,"{""2W"": 0.37, ""Car"": 0.43, ""LCV"": 0.11, ""HCV"": ...",...,NaN,NaN,NaN,NaN,NaN,0,No Holiday,0,No Event,0
1,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-03-23,NaN,308.0,308,"{""2W"": 0.43, ""Car"": 0.39, ""LCV"": 0.08, ""HCV"": ...",...,NaN,NaN,NaN,NaN,NaN,0,No Holiday,0,No Event,0
2,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,NaN,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",...,WS-NORTH,clear,16.3,0.0,10000.0,0,No Holiday,0,No Event,0
3,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,NaN,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",...,WS-NORTH,Clear,14.7,0.0,10000.0,0,No Holiday,0,No Event,0
4,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,NaN,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",...,WS-NORTH,Clear,16.7,0.0,10000.0,0,No Holiday,0,No Event,0


In [10]:
print("Traffic original time:")
print(traffic_df["time"].head(10).tolist())
print("Traffic time type:", traffic_df["time"].dtype)

print("\nWeather original time:")
print(weather_df["time"].head(10).tolist())
print("Weather time type:", weather_df["time"].dtype)

print("\nTraffic rows:", len(traffic_df))
print("Merged rows:", len(merged_df))

Traffic original time:
['07:30', '21:00', '11:00', '01:30', '12:30', '10:00', '00:00', '23:00', '17:00', '06:30']
Traffic time type: object

Weather original time:
['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00']
Weather time type: object

Traffic rows: 178468
Merged rows: 858164


In [11]:
import pandas as pd
import numpy as np

# 1. Load the three datasets
traffic_df = pd.read_csv(BASE_DIR / "data" / "traffic_sensor_log.csv")
weather_df = pd.read_csv(BASE_DIR / "data" / "weather_observations.csv")
calendar_df = pd.read_csv(BASE_DIR / "data" / "calendar_events.csv")

# 2. Clean column names
for df in [traffic_df, weather_df, calendar_df]:
    df.columns = df.columns.str.strip()

# 3. Convert traffic date and time
traffic_df["date"] = pd.to_datetime(
    traffic_df["date"], errors="coerce"
).dt.date

traffic_df["datetime"] = pd.to_datetime(
    traffic_df["date"].astype(str) + " " + traffic_df["time"].astype(str),
    errors="coerce"
)

# 4. Convert weather date and time
weather_df["date"] = pd.to_datetime(
    weather_df["date"], dayfirst=True, errors="coerce"
).dt.date

weather_df["datetime"] = pd.to_datetime(
    weather_df["date"].astype(str) + " " + weather_df["time"].astype(str),
    errors="coerce"
)

# Weather is hourly; align traffic to the same hour
traffic_df["weather_hour"] = traffic_df["datetime"].dt.floor("h")
weather_df["weather_hour"] = weather_df["datetime"].dt.floor("h")

# 5. Standardise weather station IDs
traffic_df["weather_station_id"] = (
    traffic_df["weather_station_id"].astype(str).str.strip().str.upper()
)
weather_df["station_id"] = (
    weather_df["station_id"].astype(str).str.strip().str.upper()
)

# 6. Remove duplicate weather keys before merging
weather_df = weather_df.drop_duplicates(
    subset=["station_id", "weather_hour"]
)

# 7. Merge weather data
merged_df = traffic_df.merge(
    weather_df,
    left_on=["weather_station_id", "weather_hour"],
    right_on=["station_id", "weather_hour"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_weather")
)

# 8. Merge calendar data
calendar_df["date"] = pd.to_datetime(
    calendar_df["date"], errors="coerce"
).dt.date

calendar_df = calendar_df.drop_duplicates(subset=["date"])

merged_df = merged_df.merge(
    calendar_df,
    on="date",
    how="left",
    validate="many_to_one",
    suffixes=("", "_calendar")
)

# 9. Check final result
print("Original traffic rows:", len(traffic_df))
print("Final merged rows:", len(merged_df))

print("\nMissing values:")
print(merged_df.isnull().sum())

print("\nFirst 5 rows:")
display(merged_df.head())

Original traffic rows: 178468
Final merged rows: 178468

Missing values:
road_id                    0
road_name                  0
latitude                   0
longitude                  0
weather_station_id         0
date                       0
time                       0
traffic_volume          4387
vehicle_count              0
vehicle_type_dist          0
avg_speed               4382
occupancy               4383
congestion_level       26883
travel_time                0
accident_count             0
signal_timing              0
road_capacity              0
datetime                   0
weather_hour               0
station_id                 0
date_weather               0
time_weather               0
weather_condition          0
temperature             2744
rainfall                   0
visibility              1826
datetime_weather           0
public_holiday             0
holiday_name          171385
event_flag                 0
event_name            171360
roadwork_flag              0

,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,...,weather_condition,temperature,rainfall,visibility,datetime_weather,public_holiday,holiday_name,event_flag,event_name,roadwork_flag
0,NL-007,Kingsway @ Park,28.647903,77.149142,WS-NORTH,2025-02-26,07:30,706.0,706,"{""2W"": 0.37, ""Car"": 0.43, ""LCV"": 0.11, ""HCV"": ...",...,rain,14.8,8.5,4476.0,2025-02-26 07:00:00,0,NaN,0,NaN,0
1,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-03-23,21:00,308.0,308,"{""2W"": 0.43, ""Car"": 0.39, ""LCV"": 0.08, ""HCV"": ...",...,Clear,18.6,0.0,10000.0,2025-03-23 21:00:00,0,NaN,0,NaN,0
2,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,11:00,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",...,Clear,20.6,0.0,10000.0,2025-04-05 11:00:00,0,NaN,0,NaN,0
3,NL-006,Central Flyover E,28.622342,77.258257,WS-SOUTH,2025-05-04,01:30,187.0,187,"{""2W"": 0.41, ""Car"": 0.39, ""LCV"": 0.11, ""HCV"": ...",...,Cloudy,17.8,0.0,10000.0,2025-05-04 01:00:00,0,NaN,0,NaN,0
4,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-01-08,12:30,490.0,490,"{""2W"": 0.39, ""Car"": 0.4, ""LCV"": 0.1, ""HCV"": 0.11}",...,clear,19.1,0.0,10000.0,2025-01-08 12:00:00,0,NaN,0,NaN,0


In [12]:
print("Original traffic rows:", len(traffic_df))
print("Merged rows:", len(merged_df))

print("Traffic duplicate rows:", traffic_df.duplicated().sum())
print("Merged duplicate rows:", merged_df.duplicated().sum())

print("Traffic columns:", traffic_df.shape[1])
print("Merged columns:", merged_df.shape[1])

Original traffic rows: 178468
Merged rows: 178468
Traffic duplicate rows: 1767
Merged duplicate rows: 1767
Traffic columns: 19
Merged columns: 32


In [13]:
print("Traffic shape:", traffic_df.shape)
print("Merged shape:", merged_df.shape)

print("\nTraffic duplicate rows:")
print(traffic_df.duplicated().sum())

print("\nMerged duplicate rows:")
print(merged_df.duplicated().sum())

print("\nMerge code: check whether you used how='inner' or how='left'")

Traffic shape: (178468, 19)
Merged shape: (178468, 32)

Traffic duplicate rows:
1767

Merged duplicate rows:
1767

Merge code: check whether you used how='inner' or how='left'


In [14]:
duplicate_keys = traffic_df[
    traffic_df.duplicated(
        subset=["road_id", "date", "time"],
        keep=False
    )
].sort_values(["road_id", "date", "time"])

print("Duplicate road/time records:", len(duplicate_keys))
display(duplicate_keys.head(10))

Duplicate road/time records: 3534


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,avg_speed,occupancy,congestion_level,travel_time,accident_count,signal_timing,road_capacity,datetime,weather_hour
137573,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-04,06:00,240.0,240,"{""2W"": 0.43, ""Car"": 0.35, ""LCV"": 0.11, ""HCV"": ...",55.2,15.8,Free-flow,2.16,0,36,1888,2025-01-04 06:00:00,2025-01-04 06:00:00
147678,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-04,06:00,240.0,240,"{""2W"": 0.43, ""Car"": 0.35, ""LCV"": 0.11, ""HCV"": ...",55.2,15.8,Free-flow,2.16,0,36,1888,2025-01-04 06:00:00,2025-01-04 06:00:00
19191,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,05:00,139.0,139,"{""2W"": 0.4, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": 0.1}",52.7,8.9,NaN,2.27,0,43,1888,2025-01-11 05:00:00,2025-01-11 05:00:00
110142,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,05:00,139.0,139,"{""2W"": 0.4, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": 0.1}",52.7,8.9,NaN,2.27,0,43,1888,2025-01-11 05:00:00,2025-01-11 05:00:00
59694,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,09:30,NaN,641,"{""2W"": 0.38, ""Car"": 0.44, ""LCV"": 0.11, ""HCV"": ...",NaN,NaN,Moderate,2.96,0,59,1888,2025-01-11 09:30:00,2025-01-11 09:00:00
70453,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,09:30,NaN,641,"{""2W"": 0.38, ""Car"": 0.44, ""LCV"": 0.11, ""HCV"": ...",NaN,NaN,Moderate,2.96,0,59,1888,2025-01-11 09:30:00,2025-01-11 09:00:00
26562,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-12,06:00,208.0,208,"{""2W"": 0.38, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": ...",50.9,13.5,Free-flow,2.35,0,40,1888,2025-01-12 06:00:00,2025-01-12 06:00:00
36768,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-12,06:00,208.0,208,"{""2W"": 0.38, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": ...",50.9,13.5,Free-flow,2.35,0,40,1888,2025-01-12 06:00:00,2025-01-12 06:00:00
152380,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-14,05:30,224.0,224,"{""2W"": 0.42, ""Car"": 0.39, ""LCV"": 0.09, ""HCV"": ...",51.3,16.4,NaN,2.33,0,39,1888,2025-01-14 05:30:00,2025-01-14 05:00:00
152897,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-14,05:30,224.0,224,"{""2W"": 0.42, ""Car"": 0.39, ""LCV"": 0.09, ""HCV"": ...",51.3,16.4,NaN,2.33,0,39,1888,2025-01-14 05:30:00,2025-01-14 05:00:00


In [15]:
# Remove only completely identical records
merged_df = merged_df.drop_duplicates(keep="first").copy()

print("Final dataset shape:", merged_df.shape)
print("Remaining exact duplicates:", merged_df.duplicated().sum())

# Save cleaned merged dataset
output_path = BASE_DIR / "data" / "flowcast_merged_cleaned.csv"
merged_df.to_csv(output_path, index=False)

print("Saved successfully:", output_path)

Final dataset shape: (176701, 32)
Remaining exact duplicates: 0
Saved successfully: C:\Users\pathi\Desktop\FlowCast\data\flowcast_merged_cleaned.csv


In [16]:
import pandas as pd
from pathlib import Path

# Load the cleaned merged dataset
file_path = BASE_DIR / "data" / "flowcast_merged_cleaned.csv"

merged_df = pd.read_csv(file_path)

print("Dataset shape:", merged_df.shape)

# Missing-value report
missing_report = pd.DataFrame({
    "Missing_Count": merged_df.isnull().sum(),
    "Missing_Percentage": (
        merged_df.isnull().mean() * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values("Missing_Count", ascending=False)

print("\nMissing Values Report:")
display(missing_report)

Dataset shape: (176701, 32)

Missing Values Report:


,Missing_Count,Missing_Percentage
holiday_name,169690,96.03
event_name,169663,96.02
congestion_level,26624,15.07
traffic_volume,4348,2.46
occupancy,4344,2.46
avg_speed,4343,2.46
temperature,2708,1.53
visibility,1806,1.02


In [17]:
merged_df["holiday_name"] = (
    merged_df["holiday_name"].fillna("No Holiday")
)

merged_df["event_name"] = (
    merged_df["event_name"].fillna("No Event")
)

print("Remaining missing values:")
print(merged_df[["holiday_name", "event_name"]].isnull().sum())

Remaining missing values:
holiday_name    0
event_name      0
dtype: int64


In [3]:
check_cols = [
    "traffic_volume",
    "occupancy",
    "avg_speed",
    "temperature",
    "visibility",
    "congestion_level"
]

print(merged_df[check_cols].isnull().sum())

traffic_volume       4348
occupancy            4344
avg_speed            4343
temperature          2708
visibility           1806
congestion_level    26624
dtype: int64


In [2]:
import pandas as pd

merged_df = pd.read_csv(
    r"C:\Users\pathi\Desktop\FlowCast\data\flowcast_merged_cleaned.csv"
)

print("Dataset loaded:", merged_df.shape)

Dataset loaded: (176701, 32)


In [4]:
numeric_cols = [
    "traffic_volume",
    "occupancy",
    "avg_speed",
    "temperature",
    "visibility"
]

for col in numeric_cols:
    merged_df[col] = merged_df.groupby("road_id")[col].transform(
        lambda x: x.fillna(x.median())
    )

print("Remaining missing values:")
print(merged_df[numeric_cols].isnull().sum())

Remaining missing values:
traffic_volume    0
occupancy         0
avg_speed         0
temperature       0
visibility        0
dtype: int64


In [5]:
# Keep only rows with known congestion labels
congestion_df = merged_df.dropna(
    subset=["congestion_level"]
).copy()

print("Original dataset:", merged_df.shape)
print("Congestion dataset:", congestion_df.shape)
print("Missing congestion labels:",
      congestion_df["congestion_level"].isnull().sum())

Original dataset: (176701, 32)
Congestion dataset: (150077, 32)
Missing congestion labels: 0


In [6]:
from pathlib import Path

DATA_DIR = Path(
    r"C:\Users\pathi\Desktop\FlowCast\data"
)

# Save complete traffic dataset
merged_df.to_csv(
    DATA_DIR / "flowcast_traffic_cleaned.csv",
    index=False
)

# Save congestion modelling dataset
congestion_df.to_csv(
    DATA_DIR / "flowcast_congestion_cleaned.csv",
    index=False
)

print("Both datasets saved successfully!")

Both datasets saved successfully!


In [7]:
import joblib

model = joblib.load("../models/accident_risk_model.joblib")

print("Model:", type(model))
print("Expected features:", getattr(model, "n_features_in_", "Not available"))
print("Feature names:", getattr(model, "feature_names_in_", "Not available"))

Model: <class 'dict'>
Expected features: Not available
Feature names: Not available


In [8]:
print("Dictionary keys:", model.keys())

for key, value in model.items():
    print("\nKey:", key)
    print("Type:", type(value))

    if hasattr(value, "n_features_in_"):
        print("Expected features:", value.n_features_in_)

    if hasattr(value, "feature_names_in_"):
        print("Feature names:", value.feature_names_in_)

Dictionary keys: dict_keys(['model', 'imputer', 'features', 'threshold'])

Key: model
Type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Expected features: 32

Key: imputer
Type: <class 'sklearn.impute._base.SimpleImputer'>
Expected features: 32
Feature names: ['latitude' 'longitude' 'traffic_volume' 'vehicle_count' 'avg_speed'
 'occupancy' 'travel_time' 'signal_timing' 'road_capacity'
 'capacity_30min' 'volume_capacity_ratio' 'temperature' 'rainfall'
 'visibility' 'public_holiday' 'event_flag' 'roadwork_flag' 'hour'
 'minute' 'day_of_week' 'day_of_month' 'month' 'is_weekend'
 'is_morning_peak' 'is_evening_peak' 'traffic_volume_lag_1'
 'traffic_volume_lag_2' 'traffic_volume_lag_4' 'avg_speed_lag_1'
 'avg_speed_lag_2' 'traffic_volume_rolling_mean_4'
 'avg_speed_rolling_mean_4']

Key: features
Type: <class 'list'>

Key: threshold
Type: <class 'float'>


In [9]:
print("ACCIDENT MODEL FEATURES:")
for i, feature in enumerate(model["features"], start=1):
    print(i, feature)

print("\nThreshold:", model["threshold"])

ACCIDENT MODEL FEATURES:
1 latitude
2 longitude
3 traffic_volume
4 vehicle_count
5 avg_speed
6 occupancy
7 travel_time
8 signal_timing
9 road_capacity
10 capacity_30min
11 volume_capacity_ratio
12 temperature
13 rainfall
14 visibility
15 public_holiday
16 event_flag
17 roadwork_flag
18 hour
19 minute
20 day_of_week
21 day_of_month
22 month
23 is_weekend
24 is_morning_peak
25 is_evening_peak
26 traffic_volume_lag_1
27 traffic_volume_lag_2
28 traffic_volume_lag_4
29 avg_speed_lag_1
30 avg_speed_lag_2
31 traffic_volume_rolling_mean_4
32 avg_speed_rolling_mean_4

Threshold: 0.1
